In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D 
from matplotlib import gridspec


import os


import seaborn as sns
import pysam as ps



## Load data

### Load cell-calling methods' cell-calling results 

In [2]:
# Load Entropy value for each barcode on atac 
entropy_dir = '/mnt/hdd_bob/syy/adipose/atac/res/VIB_10xmultiome_2_WS3000F'
cell_calling_comparison_dir = os.path.join(entropy_dir, '_cell_calling_comparison')

Union_3set_atac_BCs_file = os.path.join(cell_calling_comparison_dir, '_Union_cell_3set_with_RNA_cluster_and_DNAdebrisflag.tsv')
Union_3set_atac_BCs_df = pd.read_csv(Union_3set_atac_BCs_file, sep='\t')

# Load  atac-rna-barcode map 
bc_map_file = '/home/syyang/adipose_ln/multiom/atac_rna_barcodes_map.tsv'
bc_map_pd = pd.read_csv(bc_map_file, sep='\t')



In [5]:
Union_3set_atac_BCs_df.head(2)

,atac_bc,total_fragments,atac_pass_CR,atac_pass_entropy,atac_pass_archr_TSS,_2set_identified_by_atac_SC,_2set_identified_by_atac_ST,rna_barcodes,rna_pass_CR,Entropy,leiden_RNA_scVI,DNA_debris,atac_bc-1
0,AAACAAGCAAGTTAGT,2412,True,True,True,both,both,GGCTAGTGTGTTAGCA,True,0.218328,3,NO,AAACAAGCAAGTTAGT-1
1,AAACAAGCATTTCTTC,9030,True,True,True,both,both,GGCTAGTGTACGTTTC,True,0.074451,0,NO,AAACAAGCATTTCTTC-1


In [6]:
Union_3set_atac_BCs_df['DNA_debris'].value_counts()

DNA_debris
NO     2362
YES      22
Name: count, dtype: int64

### Load fragment file that have reads correspoinding to the Union BCs

In [3]:
union_bc_subdir = os.path.join(entropy_dir, '_cell_calling_comparison')

union_bc_sam_file = os.path.join(union_bc_subdir, 'sorted_tagged_union_bc_fragments.bam')




In [4]:
com_dir = os.path.join(union_bc_subdir, 'cmp_DNAdebris_entropy')
os.makedirs(com_dir, exist_ok=True)

DNAdebris_bc_bam_file = os.path.join(com_dir, 'DNAdebris_fragments.bam')
not_DNAdebris_bc_bam_file = os.path.join(com_dir, 'NotDNAdebris_fragments.bam')



In [8]:
for out, id in zip([DNAdebris_bc_bam_file, not_DNAdebris_bc_bam_file], 
                   [flag + '_dnadebris' for flag in ['YES', 'NO']]):
    sam = ps.AlignmentFile(union_bc_sam_file, 'rb')
    outfile = ps.AlignmentFile(out, "wb", template=sam)
    for read in sam.fetch(until_eof=True):
        #  if the read has the tag id
        if read.get_tag('DR') == id:
                outfile.write(read)

    outfile.close()
    sam.close()